# Sentiment Analysis dengan Word Embedding (GloVe) + Bidirectional LSTM

Klasifikasi sentimen review film (IMDB) menggunakan **pretrained word embedding (GloVe)** sebagai representasi kata, dan **Bidirectional LSTM** sebagai encoder sekuensial.

**Alur notebook:** Setup → Dataset → Vocabulary & Word Embedding → Model → Training → Evaluasi → Inference → Simpan Model.


## 1. Setup & Dependencies

In [ ]:
!pip install torchtext==0.6

In [ ]:
import torch
from torchtext import data

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# TEXT: field untuk kalimat, di-tokenize pakai spaCy
TEXT = data.Field(tokenize='spacy', tokenizer_language='en_core_web_sm', include_lengths=True)
# LABEL: field untuk label sentimen (0/1)
LABEL = data.LabelField(dtype=torch.float)

## 2. Dataset: IMDB Movie Reviews

Dataset berisi 50.000 review film berlabel positif/negatif, dibagi jadi train dan test set.

In [ ]:
from torchtext import datasets

train_data, test_data = datasets.IMDB.splits(TEXT, LABEL)

import random
train_data, valid_data = train_data.split(random_state=random.seed(SEED))

## 3. Vocabulary & Word Embedding (GloVe)

Alih-alih melatih embedding dari nol, kita pakai **pretrained GloVe embedding** (`glove.6B.100d` — 100 dimensi, dilatih dari 6 miliar token). Setiap kata di vocabulary akan dipetakan ke vektor 100 dimensi yang sudah menangkap kemiripan makna antar kata.

Referensi vocab/GloVe: [glove6b100dtxt on Kaggle](https://www.kaggle.com/datasets/danielwillgeorge/glove6b100dtxt)

In [ ]:
MAX_VOCAB_SIZE = 25_000

TEXT.build_vocab(train_data,
                  max_size=MAX_VOCAB_SIZE,
                  vectors='glove.6B.100d',
                  unk_init=torch.Tensor.normal_)

LABEL.build_vocab(train_data)

In [ ]:
BATCH_SIZE = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data),
    batch_size=BATCH_SIZE,
    sort_within_batch=True,
    device=device)

## 4. Model: Bidirectional LSTM + Embedding Layer

> **Catatan:** di source aslinya class ini dinamai `RNN`, tapi implementasinya memakai `nn.LSTM` (bukan vanilla RNN) — di sini saya beri nama `SentimentLSTM` supaya lebih jelas dan tidak membingungkan.

Arsitektur: `Embedding → Bidirectional LSTM (2 layer) → Dropout → Linear`. Hidden state dari arah forward dan backward pada layer terakhir digabung sebelum masuk ke classifier.

In [ ]:
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.rnn = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=n_layers,
                            bidirectional=bidirectional,
                            dropout=dropout)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        # text = [sent len, batch size]
        embedded = self.dropout(self.embedding(text))
        # embedded = [sent len, batch size, emb dim]

        # pack sequence supaya LSTM tidak memproses padding
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, text_lengths.to('cpu'))
        packed_output, (hidden, cell) = self.rnn(packed_embedded)

        # gabungkan hidden state terakhir dari arah forward & backward
        hidden = self.dropout(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        # hidden = [batch size, hid dim * num directions]

        return self.fc(hidden)

In [ ]:
INPUT_DIM = len(TEXT.vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
PAD_IDX = TEXT.vocab.stoi[TEXT.pad_token]

model = SentimentLSTM(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM,
                       N_LAYERS, BIDIRECTIONAL, DROPOUT, PAD_IDX)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model memiliki {count_parameters(model):,} trainable parameters')

Model memiliki 4,810,857 trainable parameters


## 5. Memuat Bobot GloVe ke Embedding Layer

In [ ]:
pretrained_embeddings = TEXT.vocab.vectors
print(pretrained_embeddings.shape)

torch.Size([25002, 100])


In [ ]:
model.embedding.weight.data.copy_(pretrained_embeddings)

In [ ]:
# token <unk> dan <pad> di-nol-kan supaya tidak ikut mempengaruhi makna
UNK_IDX = TEXT.vocab.stoi[TEXT.unk_token]

model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model.embedding.weight.data[PAD_IDX] = torch.zeros(EMBEDDING_DIM)

## 6. Konfigurasi Training

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
def binary_accuracy(preds, y):
    """Akurasi per batch (0.0-1.0), bukan jumlah prediksi benar."""
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    return correct.sum() / len(correct)

def train(model, iterator, optimizer, criterion):
    epoch_loss, epoch_acc = 0, 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        text, text_lengths = batch.text
        predictions = model(text, text_lengths).squeeze(1)
        loss = criterion(predictions, batch.label)
        acc = binary_accuracy(predictions, batch.label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    epoch_loss, epoch_acc = 0, 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text, text_lengths = batch.text
            predictions = model(text, text_lengths).squeeze(1)
            loss = criterion(predictions, batch.label)
            acc = binary_accuracy(predictions, batch.label)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

## 7. Training (5 Epoch)

In [ ]:
import time

def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    mins = int(elapsed / 60)
    secs = int(elapsed - mins * 60)
    return mins, secs

N_EPOCHS = 5
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion)

    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'sentiment-model.pt')

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Epoch: 01 | Epoch Time: 0m 34s
	Train Loss: 0.658 | Train Acc: 60.43%
	 Val. Loss: 0.540 |  Val. Acc: 72.92%
Epoch: 02 | Epoch Time: 0m 34s
	Train Loss: 0.556 | Train Acc: 71.12%
	 Val. Loss: 0.440 |  Val. Acc: 80.07%
Epoch: 03 | Epoch Time: 0m 35s
	Train Loss: 0.417 | Train Acc: 81.63%
	 Val. Loss: 0.341 |  Val. Acc: 85.54%
Epoch: 04 | Epoch Time: 0m 37s
	Train Loss: 0.321 | Train Acc: 87.10%
	 Val. Loss: 0.327 |  Val. Acc: 86.75%
Epoch: 05 | Epoch Time: 0m 36s
	Train Loss: 0.284 | Train Acc: 88.73%
	 Val. Loss: 0.295 |  Val. Acc: 88.00%


## 8. Evaluasi di Test Set

In [ ]:
model.load_state_dict(torch.load('sentiment-model.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 0.303 | Test Acc: 87.75%


## 9. Inference: Coba Kalimat Baru

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

def predict_sentiment(model, sentence):
    model.eval()
    tokenized = [tok.text for tok in nlp.tokenizer(sentence)]
    indexed = [TEXT.vocab.stoi[t] for t in tokenized]
    length = [len(indexed)]
    tensor = torch.LongTensor(indexed).to(device)
    tensor = tensor.unsqueeze(1)
    length_tensor = torch.LongTensor(length)
    with torch.no_grad():
        prediction = torch.sigmoid(model(tensor, length_tensor))
    return prediction.item()

In [ ]:
predict_sentiment(model, 'This film is terrible')

0.0055189854465425014

In [ ]:
predict_sentiment(model, 'This film is great')

0.9846353530883789

## 10. Simpan Model & Vocabulary ke Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

directory = r'/content/drive/MyDrive/AI-Engineer/NLP/models/'
os.makedirs(directory, exist_ok=True)

save_path = os.path.join(directory, 'sentiment-model.pt')
torch.save(model.state_dict(), save_path)
print(f'Model berhasil disimpan ke {save_path}')

In [ ]:
# Vocabulary juga disimpan supaya bisa dipakai ulang tanpa perlu membangun ulang dari dataset
vocab_file_path = os.path.join(directory, 'vocab.txt')
with open(vocab_file_path, 'w') as vocab_file:
    for word, index in TEXT.vocab.stoi.items():
        vocab_file.write(f'{word}\t{index}\n')
print(f'Vocabulary disimpan ke {vocab_file_path}')

---

*Notebook ini disusun sebagai bagian dari sesi rubythalib.ai AI Engineer Bootcamp, dengan bimbingan mentor Muhammad Ikhwan Fathulloh. Dataset: [IMDB Movie Reviews](https://ai.stanford.edu/~amaas/data/sentiment/) (via torchtext). Word embedding: [GloVe 6B 100d](https://www.kaggle.com/datasets/danielwillgeorge/glove6b100dtxt).*